# Chapter 11 — Capstone: Full System Integration

**Multi-Agent Analog EDA — PhD-Level Capstone Notebook**

---

This chapter **integrates** the full pedagogical stack: **Stage A** (spec → structured inventory, Ch.10A), **Stage B** (inventory → SPICE netlist, Ch.10B), **Stage C** (netlist → simulated metrics & waveforms, mock oracle), and **Stage D** (metrics-aware **abstract layout** synthesis).
A **Supervisor** agent orchestrates the pipeline using the **stateful graph pattern** from **Chapter 3** (conditional routing, checkpointing, bounded retries).

### Capstone learning objectives

1. **Systems view:** articulate end-to-end *dataflow*, *state channels*, and *failure modes* for agentic analog EDA.
2. **Graph orchestration:** implement a **compiled workflow** with guards $\phi: \mathcal{X} \to \{\text{next node}\}$, **exponential backoff** retries, and **hard iteration ceilings**.
3. **Observability:** persist **checkpoints** between stages for audit, resume, and regression.
4. **Evaluation:** quantify **design quality**, **convergence**, and **throughput** vs. manual baselines on a **multi-spec benchmark**.
5. **Production bridge:** map the demo to **Chapter 9’s Twelve Pillars**—containers, traces, farm scaling, governance.

> **Environment.** Stdlib + NumPy + Matplotlib + Plotly only. **No external APIs.** Simulators and P&R are **pedagogical mocks** with interfaces ready for Ngspice/KLayout swaps.

---

In [ ]:
import sys; sys.path.insert(0, '..')
from style_utils import (setup_3b1b_style, glow_line, glow_fill, styled_box,
                         styled_arrow, finish_plot, plotly_3b1b_layout,
                         BACKGROUND, SURFACE, TEXT, TEXT_DIM, GRID,
                         BLUE, TEAL, GREEN, YELLOW, GOLD, RED,
                         ROSE, PURPLE, CYAN, ORANGE, PALETTE)
setup_3b1b_style()

from __future__ import annotations

import hashlib
import json
import math
import re
import textwrap
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

pio.templates.default = "plotly_dark"

# --- Dark theme: matplotlib #0d1117, plotly plotly_dark ---
DARK_BG = "#0d1117"
FG = "#c9d1d9"
MUTED = "#8b949e"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
AMBER = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"
CYAN = "#39d0d0"
ORANGE = "#ffa657"

MPL_RC = {
    "figure.facecolor": DARK_BG,
    "axes.facecolor": DARK_BG,
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": FG,
    "text.color": FG,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "grid.color": "#21262d",
    "grid.alpha": 0.65,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.size": 11,
}
mpl.rcParams.update(MPL_RC)

RNG = np.random.default_rng(2027)
print("Capstone environment ready — dark matplotlib + plotly_dark; RNG seed 2027.")

## 1. System architecture

### 1.1 Component graph (conceptual)

We model the autonomous design system as a **controlled transition system** on shared **pipeline state** $x \in \mathcal{X}$. Worker agents implement maps $F_A, F_B, F_C, F_D$. The **Supervisor** implements routing guards and recovery.

```mermaid
flowchart LR
  U[User NL Spec] --> A[Stage A: Spec Parser]
  A --> B[Stage B: Netlist Gen]
  B --> C[Stage C: Sim + Waveforms]
  C -->|metrics OK| D[Stage D: Layout Synth]
  C -->|refine| B
  A -->|retry/backoff| A
  D --> P[Design Package]
```

### 1.2 State management & error handling

| Channel | Role | Checkpoint? |
|--------|------|-------------|
| `raw_spec` | User intent (text) | yes (hash + text) |
| `inventory` | Structured `component_inventory/v1` | yes |
| `netlist` | SPICE string + metadata | yes |
| `metrics` | FoMs (gain, GBW, PM, …) | yes |
| `waveforms` | AC magnitude / phase samples | yes (summary) |
| `layout` | Abstract floorplan + routing sketch | yes |
| `trace` | Span log for observability | append-only |

**Recovery strategies**

1. **Transient faults:** bounded retries with **exponential backoff** on stage entry.
2. **Specification drift:** validate Stage A JSON; fail fast with actionable errors.
3. **Performance shortfall:** **C → B** refinement loop on sizing $\theta$ until slack or **max refinement**.
4. **Hard stop:** global step counter prevents infinite cycles.

---

In [ ]:
# --- Architecture diagram: full multi-agent stack (matplotlib) ---

fig, ax = plt.subplots(figsize=(13, 6.2), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis("off")
ax.set_title(
    "Capstone: Supervisor-orchestrated pipeline (A → B → C ⇄ B → D)",
    color=FG,
    fontsize=13,
    pad=12,
)

boxes = {
    "User": (0.6, 3.5),
    "Supervisor": (3.2, 3.5),
    "A: Parse": (6.0, 5.5),
    "B: Netlist": (6.0, 3.5),
    "C: Sim": (6.0, 1.5),
    "D: Layout": (9.8, 3.5),
    "Package": (12.5, 3.5),
}


def draw_box(name, xy, w=1.55, h=0.85, fc="#58a6ff22", ec=ACCENT):
    x, y = xy
    ax.add_patch(
        mpl.patches.FancyBboxPatch(
            (x - w / 2, y - h / 2),
            w,
            h,
            boxstyle="round,pad=0.03,rounding_size=0.08",
            facecolor=fc,
            edgecolor=ec,
            linewidth=1.8,
        )
    )
    ax.text(x, y, name, ha="center", va="center", fontsize=9.5, color=FG)


for k, v in boxes.items():
    draw_box(k, v)


def arrow(p, q, label=None, color=MUTED):
    ax.annotate(
        "",
        xy=q,
        xytext=p,
        arrowprops=dict(arrowstyle="->", color=color, lw=1.5, shrinkA=6, shrinkB=6),
    )
    if label:
        mx, my = (p[0] + q[0]) / 2, (p[1] + q[1]) / 2
        ax.text(mx, my + 0.22, label, ha="center", fontsize=8, color=ORANGE)


arrow(boxes["User"], boxes["Supervisor"])
arrow(boxes["Supervisor"], boxes["A: Parse"])
arrow(boxes["A: Parse"], boxes["B: Netlist"])
arrow(boxes["B: Netlist"], boxes["C: Sim"])
arrow(boxes["C: Sim"], boxes["D: Layout"], label="if spec met")
ax.annotate(
    "",
    xy=(boxes["B: Netlist"][0] + 0.2, boxes["B: Netlist"][1] + 0.5),
    xytext=(boxes["C: Sim"][0] + 0.2, boxes["C: Sim"][1] + 0.5),
    arrowprops=dict(arrowstyle="->", color=AMBER, lw=1.4, connectionstyle="arc3,rad=-0.35"),
)
ax.text(7.35, 2.85, "refine θ", color=AMBER, fontsize=8)
arrow(boxes["D: Layout"], boxes["Package"], color=GREEN)

ax.text(3.2, 2.25, "checkpoint\nafter each stage", ha="center", fontsize=8, color=CYAN)
plt.tight_layout()
plt.show()

## 2. Pipeline state & checkpointing

We use a **dataclass** with **explicit versioning** (`schema_version`) so checkpoints remain replayable. Checkpoints store **SHA-256** fingerprints of artifacts for **regression diffing**. The Supervisor merges **partial updates** per node—mirroring **reducer channels** in LangGraph-style designs (Chapter 3).

---

In [ ]:
@dataclass
class PipelineState:
    "Global workflow state: all pipeline channels (capstone)."
    schema_version: str = "capstone_pipeline/v1"
    run_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    raw_spec: str = ""
    inventory: Optional[Dict[str, Any]] = None
    netlist: str = ""
    netlist_meta: Dict[str, Any] = field(default_factory=dict)
    sizing_params: Dict[str, float] = field(default_factory=dict)
    metrics: Dict[str, float] = field(default_factory=dict)
    waveforms: Dict[str, Any] = field(default_factory=dict)
    layout: Dict[str, Any] = field(default_factory=dict)
    refine_count: int = 0
    global_step: int = 0
    stage_retries: Dict[str, int] = field(default_factory=dict)
    log: List[Dict[str, Any]] = field(default_factory=list)
    errors: List[str] = field(default_factory=list)
    terminal: str = ""  # "", "success", "fail"

    def fingerprint(self, key: str) -> str:
        if key == "inventory" and self.inventory is not None:
            blob = json.dumps(self.inventory, sort_keys=True).encode()
        elif key == "netlist":
            blob = self.netlist.encode()
        else:
            blob = b""
        return hashlib.sha256(blob).hexdigest()[:16] if blob else ""


class CheckpointStore:
    "In-memory checkpoint ring (swap for SQLite or object store in production)."

    def __init__(self, max_keep: int = 64) -> None:
        self.max_keep = max_keep
        self._buf: List[Dict[str, Any]] = []

    def save(self, stage: str, state: PipelineState) -> None:
        snap = {
            "stage": stage,
            "ts": time.time(),
            "run_id": state.run_id,
            "global_step": state.global_step,
            "fp_inventory": state.fingerprint("inventory"),
            "fp_netlist": state.fingerprint("netlist"),
            "metrics": dict(state.metrics),
            "refine_count": state.refine_count,
        }
        self._buf.append(snap)
        if len(self._buf) > self.max_keep:
            self._buf = self._buf[-self.max_keep :]

    def tail(self, n: int = 8) -> List[Dict[str, Any]]:
        return list(self._buf[-n:])


CHECKPOINTS = CheckpointStore()
print("PipelineState + CheckpointStore defined.")

## 3. Stage implementations (A–D)

**Simplified** versions of Chapters **10A–10D**:

- **A:** Regex + rules → `component_inventory/v1` JSON.
- **B:** Template compiler → SKY130-flavored two-stage **Miller OTA** + AC testbench.
- **C:** **Mock** AC simulator → FoMs + synthetic Bode arrays.
- **D:** **Abstract** layout (rows, area, metal stack).

---

In [ ]:
# --- Stage A: simplified spec parsing (Chapter 10A style) ---


def _find_float_unit(text: str, param_regex: str, unit: str) -> Optional[float]:
    m = re.search(rf"{param_regex}\s*([0-9]+(?:\.[0-9]+)?)\s*{re.escape(unit)}", text, re.I)
    return float(m.group(1)) if m else None


def simplified_parse_spec(raw: str) -> Dict[str, Any]:
    text = " ".join(raw.split())
    inv: Dict[str, Any] = {
        "schema": "multi_agent_analog_eda/component_inventory/v1",
        "pdk": "sky130A",
        "source_text": raw[:500],
        "topology_preferences": [],
        "electrical_targets": {},
        "validation": {"warnings": [], "complete": True},
    }
    if re.search(r"two-?stage|Miller", text, re.I):
        inv["topology_preferences"].append("miller_ota")
    if re.search(r"folded[- ]?cascode", text, re.I):
        inv["topology_preferences"].append("folded_cascode_ota")
    if not inv["topology_preferences"]:
        inv["topology_preferences"].append("miller_ota")
        inv["validation"]["warnings"].append("defaulted_topology_to_miller_ota")

    adc = _find_float_unit(text, r"gain|Adc|DC\s+gain", "dB")
    gbw = _find_float_unit(text, r"GBW|unity[- ]gain|bandwidth", "MHz")
    pm = _find_float_unit(text, r"phase\s+margin|PM", r"°|deg|degrees")

    if adc is not None:
        inv["electrical_targets"]["Adc_dB"] = adc
    if gbw is not None:
        inv["electrical_targets"]["GBW_MHz"] = gbw
    if pm is not None:
        inv["electrical_targets"]["phase_margin_deg"] = pm

    for k in ("Adc_dB", "GBW_MHz"):
        if k not in inv["electrical_targets"]:
            inv["validation"]["complete"] = False
            inv["validation"]["warnings"].append(f"missing_target:{k}")

    inv["library_hooks"] = {
        "lib_file": "${SKY130_ROOT}/libs.tech/ngspice/sky130.lib.spice",
        "corner_default": "tt",
    }
    inv["allowed_devices"] = [
        {"cell": "sky130_fd_pr__nfet_01v8", "role": "nmos"},
        {"cell": "sky130_fd_pr__pfet_01v8", "role": "pmos"},
    ]
    return inv


def validate_inventory(inv: Dict[str, Any]) -> Tuple[bool, List[str]]:
    errs: List[str] = []
    if inv.get("schema") != "multi_agent_analog_eda/component_inventory/v1":
        errs.append("bad_schema")
    et = inv.get("electrical_targets") or {}
    if "Adc_dB" not in et or "GBW_MHz" not in et:
        errs.append("incomplete_targets")
    return (len(errs) == 0, errs)


print("Stage A: simplified_parse_spec + validate_inventory ready.")

In [ ]:
# --- Stage B: simplified Miller OTA netlist (Chapter 10B style) ---


def default_sizing_params() -> Dict[str, float]:
    return {
        "W_in_um": 4.2,
        "L_in_um": 0.15,
        "W_load_um": 8.0,
        "L_load_um": 0.15,
        "W_2nd_um": 12.0,
        "L_2nd_um": 0.15,
        "ibias_uA": 40.0,
        "Cc_pF": 1.2,
    }


def generate_miller_netlist(inv: Dict[str, Any], p: Dict[str, float]) -> Tuple[str, Dict[str, Any]]:
    targets = inv.get("electrical_targets") or {}
    topo = (inv.get("topology_preferences") or ["miller_ota"])[0]
    meta = {"topology": topo, "targets": dict(targets), "params": dict(p)}
    if topo != "miller_ota":
        meta["note"] = "folded_cascode omitted in capstone; emitting miller_ota"

    W1, L1 = p["W_in_um"], p["L_in_um"]
    W2, L2 = p["W_load_um"], p["L_load_um"]
    W3, L3 = p["W_2nd_um"], p["L_2nd_um"]
    ib = p["ibias_uA"]
    cc = p["Cc_pF"]

    spice = textwrap.dedent(
        f'''
        * Capstone miller_ota — educational SKY130-style deck (mock-friendly)
        .title miller_two_stage_ota
        .include ${{SKY130_ROOT}}/libs.tech/ngspice/sky130.lib.spice
        .option TEMP=27

        .subckt miller_ota vip vin vout vdd vss
        Xmp1 net1 net1 vdd vdd sky130_fd_pr__pfet_01v8 w={W2}u l={L2}u
        Xmp2 net2 net1 vdd vdd sky130_fd_pr__pfet_01v8 w={W2}u l={L2}u
        Xmn1 net1 vip tail vss sky130_fd_pr__nfet_01v8 w={W1}u l={L1}u
        Xmn2 net2 vin tail vss sky130_fd_pr__nfet_01v8 w={W1}u l={L1}u
        Xmb tail bias vss vss sky130_fd_pr__nfet_01v8 w=2.0u l=0.3u
        Ibias vdd bias DC {ib}u
        Xmp3 vout net2 vdd vdd sky130_fd_pr__pfet_01v8 w={W3}u l={L3}u
        Xmn3 vout net2 vss vss sky130_fd_pr__nfet_01v8 w={W3}u l={L3}u
        Cc net2 vout {cc}p
        .ends miller_ota

        .subckt tb_ac vip vin vdd vss
        Vdd vdd 0 DC 1.8
        Vin vin 0 DC 0.9 AC 1.0 0
        Vcm vip 0 DC 0.9
        Xdut vip vin vo vdd vss miller_ota
        .ends tb_ac

        Xtb inp inn vdd vss tb_ac
        .ac dec 50 1e3 1e9
        .probe ac vm(vo) vp(vo)
        .end
        '''
    ).strip()

    return spice, meta


print("Stage B: generate_miller_netlist ready.")

In [ ]:
# --- Stage C: mock simulation + waveforms (Chapter 10C style) ---


def mock_ac_metrics(p: Dict[str, float], targets: Dict[str, float]) -> Dict[str, Any]:
    W1, ib, cc = p["W_in_um"], p["ibias_uA"], p["Cc_pF"]
    _ = targets  # real sim would use corner/target context
    # Scaled heuristics so refinement reaches ~60 dB / ~100 MHz from typical starting θ
    gain_db = 20 * math.log10(max(1.0, W1 * ib * 0.12))
    gbw_mhz = 1.45 * ib * math.sqrt(max(W1, 0.15)) / max(cc, 0.3)
    pm_deg = 90 - 35 * (ib / 80.0) ** 0.5 - 10 * (1.0 / max(cc, 0.2))
    itot = 2 * ib * 1.15

    freqs = np.logspace(3, 9, 400)
    w0 = 2 * math.pi * gbw_mhz * 1e6
    zeta = max(0.35, min(0.95, pm_deg / 95))
    _ = zeta  # reserve for future pole-splitting toy model
    H = (1.0 / ((1 + 1j * freqs / w0) * (1 + 1j * freqs / (3.2 * w0 + 1e-12)))) * (
        10 ** (gain_db / 20.0)
    )
    mag_db = 20 * np.log10(np.maximum(np.abs(H), 1e-20))

    return {
        "metrics": {
            "Adc_dB": float(gain_db),
            "GBW_MHz": float(gbw_mhz),
            "phase_margin_deg": float(pm_deg),
            "Itot_uA": float(itot),
        },
        "freqs_hz": freqs,
        "mag_db": mag_db,
        "phase_deg": np.degrees(np.angle(H)),
    }


def metrics_slack(m: Dict[str, float], t: Dict[str, float]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    if "Adc_dB" in t:
        out["Adc"] = (m["Adc_dB"] - t["Adc_dB"]) / max(abs(t["Adc_dB"]), 1e-6)
    if "GBW_MHz" in t:
        out["GBW"] = (m["GBW_MHz"] - t["GBW_MHz"]) / max(abs(t["GBW_MHz"]), 1e-6)
    if "phase_margin_deg" in t:
        out["PM"] = (m["phase_margin_deg"] - t["phase_margin_deg"]) / max(
            abs(t["phase_margin_deg"]), 1e-6
        )
    return out


def spec_satisfied(m: Dict[str, float], t: Dict[str, float], tol: float = 0.0) -> bool:
    if m["Adc_dB"] + 1e-9 < t.get("Adc_dB", 0) * (1 - tol):
        return False
    if m["GBW_MHz"] + 1e-9 < t.get("GBW_MHz", 0) * (1 - tol):
        return False
    if "phase_margin_deg" in t and m["phase_margin_deg"] < t["phase_margin_deg"] * (1 - tol):
        return False
    return True


print("Stage C: mock_ac_metrics + slack helpers ready.")

In [ ]:
# --- Stage D: abstract layout (Chapter 10D style) ---


def synthesize_layout_abstract(
    netlist: str,
    metrics: Dict[str, float],
    params: Dict[str, float],
) -> Dict[str, Any]:
    area_um2 = (
        120 * params.get("W_in_um", 1) * params.get("L_in_um", 0.15)
        + 200 * params.get("W_2nd_um", 1) * params.get("L_2nd_um", 0.15)
        + 80 * params.get("Cc_pF", 1)
    )
    rows = [
        {"row": 0, "devices": ["diffpair_n", "tail_bias"], "track_m2": 2},
        {"row": 1, "devices": ["load_mirror_p", "miller_cap"], "track_m2": 3},
        {"row": 2, "devices": ["second_stage_inv", "bias_fill"], "track_m2": 2},
    ]
    return {
        "schema": "abstract_layout/v1",
        "estimated_area_um2": float(area_um2),
        "aspect_ratio_wh": 1.35,
        "metal_stack": {"M1": "local", "M2": "signal", "M3": "power_grid"},
        "rows": rows,
        "drc_status": "clean_pedagogical_mock",
        "lvs_status": "sch_vs_abs_consistent",
        "ir_drop_mV_est": 4.2 + 0.01 * metrics.get("Itot_uA", 0),
        "notes": "Swap with KLayout/OASIS for tapeout",
    }


print("Stage D: synthesize_layout_abstract ready.")

## 4. Full Supervisor — graph workflow (Chapter 3)

The **Supervisor** compiles a **cyclic** graph: **C** may route back to **B** for refinement. Each stage uses:

- `max_retries_per_stage` and **exponential backoff** $b_k = \min(b_{\max}, b_0 \cdot 2^k)$,
- **global** `max_global_steps`,
- **checkpoint** writes after successful stage completion.

**Guard** $\phi_C(x)$: if `spec_satisfied` → **D**; else adjust $\theta$ and clear netlist/metrics unless `max_refine` exceeded.

---

In [ ]:
# --- Supervisor: stateful graph execution ---


def refine_params(p: Dict[str, float], slack: Dict[str, float]) -> Dict[str, float]:
    q = dict(p)
    if slack.get("Adc", 0) < 0:
        q["W_in_um"] = min(60.0, q["W_in_um"] * 1.12)
        q["W_load_um"] = min(80.0, q["W_load_um"] * 1.05)
    if slack.get("GBW", 0) < 0:
        q["ibias_uA"] = min(250.0, q["ibias_uA"] * 1.15)
        q["Cc_pF"] = max(0.35, q["Cc_pF"] * 0.92)
    if slack.get("PM", 0) < 0:
        q["Cc_pF"] = min(15.0, q["Cc_pF"] * 1.08)
        q["ibias_uA"] = max(8.0, q["ibias_uA"] * 0.95)
    return q


@dataclass
class SupervisorConfig:
    max_retries_per_stage: int = 4
    backoff_base_s: float = 0.04
    backoff_cap_s: float = 0.8
    max_global_steps: int = 200
    max_refine: int = 12


class CapstoneSupervisor:
    def __init__(self, cfg: Optional[SupervisorConfig] = None) -> None:
        self.cfg = cfg or SupervisorConfig()
        self.checkpoints = CheckpointStore()

    def _log(self, state: PipelineState, event: str, **kw: Any) -> None:
        state.global_step += 1
        state.log.append({"event": event, "step": state.global_step, **kw})

    def _retry_sleep(self, stage: str, state: PipelineState) -> None:
        k = state.stage_retries.get(stage, 0)
        delay = min(self.cfg.backoff_cap_s, self.cfg.backoff_base_s * (2**k))
        time.sleep(delay)

    def run_stage_a(self, state: PipelineState) -> None:
        stage = "A"
        for attempt in range(self.cfg.max_retries_per_stage):
            try:
                state.inventory = simplified_parse_spec(state.raw_spec)
                ok, errs = validate_inventory(state.inventory)
                if not ok:
                    raise ValueError("validation_failed:" + ",".join(errs))
                self._log(state, "stage_a_ok", fp=state.fingerprint("inventory"))
                self.checkpoints.save(stage, state)
                return
            except Exception as e:
                state.stage_retries[stage] = state.stage_retries.get(stage, 0) + 1
                state.errors.append(f"A:{e}")
                self._log(state, "stage_a_retry", attempt=attempt, err=str(e))
                if attempt + 1 == self.cfg.max_retries_per_stage:
                    state.terminal = "fail"
                    return
                self._retry_sleep(stage, state)

    def run_stage_b(self, state: PipelineState) -> None:
        stage = "B"
        assert state.inventory is not None
        if not state.sizing_params:
            state.sizing_params = default_sizing_params()
        for attempt in range(self.cfg.max_retries_per_stage):
            try:
                sp, meta = generate_miller_netlist(state.inventory, state.sizing_params)
                if len(sp) < 80:
                    raise ValueError("netlist_too_short")
                state.netlist = sp
                state.netlist_meta = meta
                self._log(state, "stage_b_ok", fp=state.fingerprint("netlist"))
                self.checkpoints.save(stage, state)
                return
            except Exception as e:
                state.stage_retries[stage] = state.stage_retries.get(stage, 0) + 1
                state.errors.append(f"B:{e}")
                self._log(state, "stage_b_retry", attempt=attempt, err=str(e))
                if attempt + 1 == self.cfg.max_retries_per_stage:
                    state.terminal = "fail"
                    return
                self._retry_sleep(stage, state)

    def run_stage_c(self, state: PipelineState) -> None:
        stage = "C"
        assert state.inventory is not None
        targets = state.inventory["electrical_targets"]
        for attempt in range(self.cfg.max_retries_per_stage):
            try:
                sim = mock_ac_metrics(state.sizing_params, targets)
                state.metrics = sim["metrics"]
                state.waveforms = {
                    "freqs_hz": sim["freqs_hz"],
                    "mag_db": sim["mag_db"],
                    "phase_deg": sim["phase_deg"],
                }
                self._log(state, "stage_c_ok", metrics=dict(state.metrics))
                self.checkpoints.save(stage, state)
                return
            except Exception as e:
                state.stage_retries[stage] = state.stage_retries.get(stage, 0) + 1
                state.errors.append(f"C:{e}")
                self._log(state, "stage_c_retry", attempt=attempt, err=str(e))
                if attempt + 1 == self.cfg.max_retries_per_stage:
                    state.terminal = "fail"
                    return
                self._retry_sleep(stage, state)

    def run_stage_d(self, state: PipelineState) -> None:
        stage = "D"
        for attempt in range(self.cfg.max_retries_per_stage):
            try:
                state.layout = synthesize_layout_abstract(
                    state.netlist, state.metrics, state.sizing_params
                )
                self._log(state, "stage_d_ok", area=state.layout["estimated_area_um2"])
                self.checkpoints.save(stage, state)
                return
            except Exception as e:
                state.stage_retries[stage] = state.stage_retries.get(stage, 0) + 1
                state.errors.append(f"D:{e}")
                self._log(state, "stage_d_retry", attempt=attempt, err=str(e))
                if attempt + 1 == self.cfg.max_retries_per_stage:
                    state.terminal = "fail"
                    return
                self._retry_sleep(stage, state)

    def run(self, raw_spec: str) -> PipelineState:
        state = PipelineState(raw_spec=raw_spec)
        self._log(state, "run_start")
        while state.terminal == "" and state.global_step < self.cfg.max_global_steps:
            if state.inventory is None:
                self.run_stage_a(state)
                if state.terminal:
                    break
                continue

            if not state.netlist:
                self.run_stage_b(state)
                if state.terminal:
                    break
                continue

            if not state.metrics:
                self.run_stage_c(state)
                if state.terminal:
                    break
                targets = state.inventory["electrical_targets"]
                if spec_satisfied(state.metrics, targets):
                    self.run_stage_d(state)
                    if state.terminal == "":
                        state.terminal = "success"
                    break
                state.refine_count += 1
                if state.refine_count > self.cfg.max_refine:
                    state.terminal = "fail"
                    state.errors.append("max_refine_exceeded")
                    break
                slack = metrics_slack(state.metrics, targets)
                state.sizing_params = refine_params(state.sizing_params, slack)
                state.netlist = ""
                state.metrics = {}
                state.waveforms = {}
                self._log(state, "route_c_to_b", refine=state.refine_count, slack=slack)
                continue

            break

        self._log(state, "run_end", terminal=state.terminal)
        return state


SUP = CapstoneSupervisor()
print("CapstoneSupervisor compiled (graph: A → B → C ⇄ B → D).")

## 5. End-to-end demo

**Input:** *Design a two-stage Miller OTA with 60dB gain, 100MHz GBW*

We show **Stage A–D** artifacts and a **Bode magnitude** plot (mock AC).

---

In [ ]:
SPEC = "Design a two-stage Miller OTA with 60dB gain, 100MHz GBW"

state_demo = CapstoneSupervisor(SupervisorConfig(max_refine=20)).run(SPEC)

print("=== Terminal status:", state_demo.terminal, "===")
print("\n--- Stage A: inventory (excerpt) ---")
js = json.dumps(state_demo.inventory, indent=2)
print(js[:1400] + ("..." if len(js) > 1400 else ""))

print("\n--- Stage B: netlist (first 32 lines) ---")
print("\n".join(state_demo.netlist.splitlines()[:32]))

print("\n--- Stage C: metrics ---")
print(json.dumps(state_demo.metrics, indent=2))

print("\n--- Stage D: abstract layout ---")
print(json.dumps(state_demo.layout, indent=2))

wf = state_demo.waveforms
if wf:
    fig, ax = plt.subplots(figsize=(10, 4), facecolor=DARK_BG)
    ax.set_facecolor(DARK_BG)
    ax.semilogx(wf["freqs_hz"], wf["mag_db"], color=ACCENT, lw=1.6, label="|H| (dB)")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.legend(facecolor="#161b22", edgecolor="#30363d")
    ax.set_title("Stage C: mock AC magnitude (waveform analysis)", color=FG)
    plt.tight_layout()
    plt.show()

PACKAGE = {
    "run_id": state_demo.run_id,
    "spec": SPEC,
    "inventory": state_demo.inventory,
    "netlist": state_demo.netlist,
    "metrics": state_demo.metrics,
    "layout": state_demo.layout,
    "checkpoints": CHECKPOINTS.tail(6),
}
print("\n=== Design package keys:", list(PACKAGE.keys()), "===")

## 6. Evaluation framework

1. **Design quality** — scalar **Q** from normalized slacks vs. targets.
2. **Convergence** — `refine_count` and supervisor **log** (Plotly).
3. **Time-to-design** — wall-clock vs. a **transparent manual-hours model** (calibrate externally).
4. **Multi-design benchmark** — batch NL specs.

---

In [ ]:
# --- Evaluation: quality, throughput, benchmark ---


def design_quality_score(m: Dict[str, float], t: Dict[str, float]) -> float:
    slack = metrics_slack(m, t)
    parts = [max(0.0, min(1.0, 0.5 + 2.0 * slack[k])) for k in slack]
    return float(np.mean(parts)) if parts else 0.0


def manual_time_hours_estimate(complexity: str) -> float:
    return {"ota": 36.0, "lna": 48.0, "pll": 120.0}.get(complexity, 40.0)


BENCHMARK_SPECS = [
    "Miller two-stage OTA, gain 55 dB, GBW 80 MHz, phase margin 55 deg",
    "Two-stage OTA with 60 dB gain and 100 MHz GBW",
    "OTA: 72 dB, GBW 12 MHz, PM 62 deg (low power)",
    "Folded-cascode hint but targets 65 dB gain and 40 MHz GBW",
    "High speed OTA 45 dB gain 200 MHz GBW",
]

results = []
wall = []
for sp in BENCHMARK_SPECS:
    t0 = time.perf_counter()
    st = CapstoneSupervisor(SupervisorConfig(max_refine=22)).run(sp)
    dt = time.perf_counter() - t0
    wall.append(dt)
    inv = st.inventory or {}
    t = inv.get("electrical_targets") or {}
    m = st.metrics
    Q = design_quality_score(m, t) if st.terminal == "success" else 0.0
    results.append(
        {
            "spec": sp[:52] + ("..." if len(sp) > 52 else ""),
            "terminal": st.terminal,
            "refine": st.refine_count,
            "Q": round(Q, 4),
            "wall_s": round(dt, 4),
            "Adc_dB": m.get("Adc_dB"),
            "GBW_MHz": m.get("GBW_MHz"),
        }
    )

manual_h = manual_time_hours_estimate("ota")
speedup = [manual_h * 3600 / max(w, 1e-9) for w in wall]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor=DARK_BG)
for ax in axes:
    ax.set_facecolor(DARK_BG)

x = np.arange(len(BENCHMARK_SPECS))
axes[0].bar(x, [r["Q"] for r in results], color=GREEN, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"B{i}" for i in range(len(BENCHMARK_SPECS))], color=MUTED)
axes[0].set_ylabel("Quality score Q")
axes[0].set_title("Multi-design benchmark: Q", color=FG)

axes[1].bar(x, speedup, color=PURPLE, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"B{i}" for i in range(len(BENCHMARK_SPECS))], color=MUTED)
axes[1].set_ylabel("× vs manual model (wall)")
axes[1].set_title(f"Throughput proxy (manual={manual_h}h OTA)", color=FG)
plt.tight_layout()
plt.show()

if state_demo.terminal == "success":
    ev = [e["step"] for e in state_demo.log if str(e.get("event", "")).startswith("stage_")]
    fig2 = go.Figure(
        go.Scatter(
            x=list(range(len(ev))),
            y=ev,
            mode="lines+markers",
            line=dict(color=CYAN),
            name="global step @ stage events",
        )
    )
    fig2.update_layout(
        title="Convergence trace: supervisor steps at stage completions (demo run)",
        paper_bgcolor=DARK_BG,
        plot_bgcolor="#161b22",
        font=dict(color=FG),
        xaxis_title="Stage-event index",
        yaxis_title="Global step",
    )
    fig2.show()

print(json.dumps(results, indent=2))

## 7. Production deployment (→ Chapter 9)

### 7.1 Containerization

Ship **pinned** images; mount **PDK** read-only at runtime; do not bake NDA files into layers.

### 7.2 Observability

**Hierarchical spans:** `supervisor.run` → `stage.B` → `tool.ngspice`. Correlate **netlist SHA**, **PDK hash**, **job id**.

### 7.3 Scaling

Priority **queues** for interactive vs batch; autoscale on **depth** and **license tokens**.

### 7.4 Twelve Pillars (Ch.9) — mapping

| # | Pillar | Capstone touchpoint |
|---|--------|---------------------|
| 1 | BYOM | Router for Stage A (LLM vs rules) |
| 2 | Observability | `log`, checkpoints |
| 3 | Context | Typed inventory JSON |
| 4 | Ontology | Device cells + topology |
| 5 | Retrieval | Netlist template library |
| 6 | Governance | RBAC on PDK; audit trail |
| 7 | CI/CD | Golden-spec regression |
| 8 | Ops / farms | Corner parallelism |
| 9 | Dev/prod | Mock vs live sim config |
| 10 | HITL | Tapeout gate |
| 11 | Docker | Reproducible worker image |
| 12 | Enterprise | Slurm/LSF batch adapters |

---

In [ ]:
DOCKERFILE_SNIPPET = r'''
# Multi-stage: analog MAS worker (illustrative)
FROM python:3.11-slim AS base
WORKDIR /app
RUN pip install --no-cache-dir numpy matplotlib plotly ipython
ENV SKY130_ROOT=/opt/pdks/sky130A
COPY agents/ /app/agents/
COPY flows/ /app/flows/
CMD ["python", "-m", "flows.supervisor_service"]
'''

print(DOCKERFILE_SNIPPET)
display(
    HTML(
        "<pre style='background:#161b22;color:#c9d1d9;padding:12px;border:1px solid #30363d;border-radius:6px'>"
        + DOCKERFILE_SNIPPET.replace("&", "&amp;").replace("<", "&lt;")
        + "</pre>"
    )
)

## 8. Course conclusion

### 8.1 Concepts across chapters

- **Ch.1–2:** AI4EDA → **agentic** loops and architectures.
- **Ch.3:** **Stateful graphs**, checkpointing, guards — **Supervisor** backbone.
- **Ch.4–7:** SOTA tools, analog fundamentals, specialized agents, physical design.
- **Ch.8:** **MCP**-style tool contracts.
- **Ch.9:** **Twelve Pillars** — production operability.
- **Ch.10:** Stage-wise identification & netlist feedback.
- **Ch.11:** **End-to-end integration** + evaluation + deployment framing.

### 8.2 Future research

**Surrogate simulators** for gradient co-design; **SMT/verification** in graph guards $\phi$; **multi-tenant** audited checkpoints; **trace-driven** policy learning for routers.

### 8.3 Vision: 2027+ agentic EDA

**Composable agent marketplaces** (sizing, matching, EM, signoff) orchestrated by **policies** trained on traces, with **HITL** accountability at tapeout.

### 8.4 Reading list

Gray/Hurst/Lewis/Meyer; Razavi; DAC/arXiv surveys on LLM+HW; **Ngspice/Xyce** docs; **KLayout** scripting. **Next step:** replace `mock_ac_metrics` with a subprocess wrapper; add a local trace exporter (Ch.9 style); run the benchmark cell in CI.

---